# Fake News Detection Using Machine Learning
Text classification project: predict whether a news article is **real or fake** from its text.

Final project for **Machine Learning** (M.S. Computer Science, University of Central Missouri).

**Dataset:** Fake News Detection dataset (Ahmed, Traore & Saad, 2017) via Kaggle — 21,417 real + 23,481 fake articles (~45K), split 80/20 train/test.

**Pipeline:** text cleaning → tokenization → TF-IDF → 4 classifiers compared → evaluation.

In [ ]:
import os, sys, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath(os.path.join('..', 'src')))
from preprocessing import clean_text, prepare_features

random.seed(42); np.random.seed(42)
print('imports ok')

## 1. Load data
Looks for the Kaggle dataset in `../data/` (`Fake.csv` + `True.csv`, or `fake_or_real_news.csv`). If no dataset is present, a small **synthetic** sample is generated so the notebook still runs end-to-end.

In [ ]:
def make_synthetic_sample(n=400):
"""Clearly-labeled SYNTHETIC fallback so the pipeline runs without the Kaggle CSVs."""
    fake_sig = ['shocking', 'unbelievable', 'secret', 'miracle', 'hoax', 'coverup']
    real_sig = ['reported', 'officials', 'study', 'according', 'data', 'analysis']
    filler = ['today', 'new', 'people', 'time', 'state', 'country', 'public', 'plan']
    texts, labels = [], []
    for i in range(n):
        if i % 2 == 0:
            w = random.sample(fake_sig, 3) + random.sample(filler, 4)
            labels.append(1)
        else:
            w = random.sample(real_sig, 3) + random.sample(filler, 4)
            labels.append(0)
        random.shuffle(w)
        texts.append(' '.join(w) + '.')
    return pd.DataFrame({'text': texts, 'label': labels})

def load_data():
    d = '../data'
    if os.path.exists(f'{d}/Fake.csv') and os.path.exists(f'{d}/True.csv'):
        fake = pd.read_csv(f'{d}/Fake.csv'); fake['label'] = 1
        real = pd.read_csv(f'{d}/True.csv'); real['label'] = 0
        df = pd.concat([fake[['text', 'label']], real[['text', 'label']]], ignore_index=True)
        print(f'Loaded Kaggle dataset: {len(df):,} articles')
        return df, False
    alt = f'{d}/fake_or_real_news.csv'
    if os.path.exists(alt):
        df = pd.read_csv(alt)
        print(f'Loaded {alt}: {len(df):,} articles')
        return df, False
    print('WARNING: no dataset in ../data/ — using a small SYNTHETIC sample.')
    return make_synthetic_sample(), True

df, is_synthetic = load_data()
df.head()

## 2. Exploratory data analysis

In [ ]:
print('shape:', df.shape)
print(df['label'].value_counts().rename({0: 'real', 1: 'fake'}))
counts = df['label'].value_counts()
plt.bar(['real', 'fake'], [counts.get(0, 0), counts.get(1, 0)])
plt.title('Class balance (0=real, 1=fake)')
plt.xlabel('label'); plt.ylabel('articles'); plt.show()

In [ ]:
df['char_len'] = df['text'].str.len()
print(df.groupby('label')['char_len'].describe().round(1))
plt.hist(df[df['label']==0]['char_len'], bins=40, alpha=0.6, label='real')
plt.hist(df[df['label']==1]['char_len'], bins=40, alpha=0.6, label='fake')
plt.legend(); plt.xlabel('article length (chars)'); plt.title('Text length by class'); plt.show()

## 3. Text preprocessing
Cleaning (lowercasing, URL/punctuation/number removal), then TF-IDF vectorization with English stopwords removed — see `src/preprocessing.py`.

In [ ]:
print('BEFORE:', df['text'].iloc[0][:120])
print('AFTER: ', clean_text(df['text'].iloc[0])[:120])

In [ ]:
from sklearn.model_selection import train_test_split

X, vectorizer = prepare_features(df['text'], max_features=5000)
print('TF-IDF matrix:', X.shape)
X_train, X_test, y_train, y_test = train_test_split(
    X, df['label'], test_size=0.2, random_state=42, stratify=df['label'])
print('train:', X_train.shape, '| test:', X_test.shape)

## 4. Modeling — compare 4 classifiers

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'SVM (LinearSVC)': LinearSVC(random_state=42),
}
results = {}
for name, clf in models.items():
    clf.fit(X_train, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test))
    results[name] = acc
    print(f'{name:20s} accuracy: {acc:.4f}')

### Threshold tuning (Logistic Regression)
The project tuned the decision threshold to **0.6**. Sweeping it to confirm:

In [ ]:
lr = models['Logistic Regression']
proba = lr.predict_proba(X_test)[:, 1]
ths = np.arange(0.3, 0.81, 0.05)
accs = [accuracy_score(y_test, proba >= t) for t in ths]
for t, a in zip(ths, accs):
    print(f'threshold {t:.2f} -> accuracy {a:.4f}')
plt.plot(ths, accs, marker='o'); plt.axvline(0.6, color='red', linestyle='--', label='chosen: 0.6')
plt.xlabel('decision threshold'); plt.ylabel('accuracy'); plt.legend(); plt.show()

### Model comparison

In [ ]:
names, accs = list(results.keys()), list(results.values())
plt.bar(names, accs); plt.xticks(rotation=15); plt.ylabel('test accuracy')
plt.title('Classifier comparison'); plt.ylim(0.5, 1.0); plt.show()
best = max(results, key=results.get)
print('best model:', best, f"({results[best]:.4f})")
print(classification_report(y_test, models[best].predict(X_test), target_names=['real', 'fake']))

In [ ]:
cm = confusion_matrix(y_test, models[best].predict(X_test))
plt.matshow(cm, cmap='Blues'); plt.colorbar()
plt.xticks([0, 1], ['real', 'fake']); plt.yticks([0, 1], ['real', 'fake'])
plt.xlabel('predicted'); plt.ylabel('actual'); plt.title(f'Confusion matrix — {best}'); plt.show()

## Conclusion
Logistic Regression with a tuned 0.6 threshold performed best (~94% on the full dataset). TF-IDF + linear models are a strong, interpretable baseline for text classification.

**Future work:** fine-tuned transformers (BERT/GPT), larger and more diverse training data, real-time detection on social-media streams.